# Part 2: Retrieval-Augmented Generation (RAG)

Team 3 CozyDay | Lin

Uses Rishabhs frozen slm_exp5.pt checkpoint.

## 1. Load the Frozen Part 1 Checkpoint

In [1]:

import torch
import torch.nn as nn
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = Path(r"D:\INFO490\model_checkpoints\slm_exp5.pt")

saved = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

chars = saved["vocab"]
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
config = saved["config"]

print("Device:", DEVICE)
print("Vocabulary:", repr("".join(chars)))
print("Architecture:", config)


Device: cpu
Vocabulary: '\n .abcdefghijklmnoprstuvwxy'
Architecture: {'embedding_dim': 96, 'num_heads': 4, 'num_layers': 2, 'feedforward_dim': 384, 'context_length': 32}


## 2. Reuse the Original Transformer Architecture

In [2]:
# ============================================================
# CELL 9A: BUILD A TRANSFORMER BLOCK
# ============================================================
# Purpose:
# A Transformer block combines:
#
# 1. Multi-head self-attention
# 2. Residual connections
# 3. Layer normalization
# 4. Feed-forward neural network
#
# Multi-head attention allows the model to learn different
# attention patterns simultaneously.
#
# Residual connections help information and gradients flow
# through the network.
#
# Layer normalization stabilizes representations.
#
# The feed-forward network transforms each token's features.
#
# PyTorch provides MultiheadAttention, which implements
# the Q/K/V projections and attention computation.
#
# We use batch_first=True:
# (batch, sequence, embedding_dimension)
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()

        # Main Component 1
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=0.0,
            batch_first=True)

        # Supporting Component 1
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

        # Main Component 2
        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, feedforward_dim),
            nn.ReLU(),
            nn.Linear(feedforward_dim, embedding_dim))


    def forward(self, x):

        T = x.shape[1]


        # True means a position is blocked from attention.
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)

        # Supporting Component 2
        # Attention + residual connection.
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)

        # Residual 1: Original representation + attention output
        x = x + attended

        # Residual 2: Updated representation + feed-forward output
        # Feed-forward network + residual connection.
        x = x + self.feedforward(self.ln2(x))

        return x

# ============================================================
# CELL 9B: BUILDING OUR COMPLETE TINY TRANSFORMER
# ============================================================
# Purpose:
# Combine all the components into one language model.
#
# Architecture:
#
# Token IDs
#    ↓
# Token Embeddings + Position Embeddings
#    ↓
# Transformer Block(s)
#    ↓
# Layer Normalization
#    ↓
# Linear Output Layer
#    ↓
# Logits for every vocabulary token
#
# The output tensor has shape:
#
# (batch_size, sequence_length, vocabulary_size)
#
# Each position produces scores for the NEXT character.
# ============================================================

class TinyTransformerLM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embedding_dim, num_heads, feedforward_dim)
            for _ in range(num_layers)
        ])

        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):

        B, T = input_ids.shape

        if T > self.context_length:
            raise ValueError("Input sequence exceeds context length.")

        positions = torch.arange(T, device=input_ids.device)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)

        return self.output(x)

## 3. Restore Trained Weights

In [3]:

model = TinyTransformerLM(
    vocab_size=len(chars),
    embedding_dim=config["embedding_dim"],
    num_heads=config["num_heads"],
    num_layers=config["num_layers"],
    feedforward_dim=config["feedforward_dim"],
    context_length=config["context_length"]
).to(DEVICE)

model.load_state_dict(saved["model_state"], strict=True)
model.eval()

parameter_count = sum(p.numel() for p in model.parameters())
assert parameter_count == 232155
assert model.context_length == 32

print("Checkpoint loaded successfully!")
print("Parameters:", parameter_count)
print("Context length:", model.context_length)


Checkpoint loaded successfully!
Parameters: 232155
Context length: 32


In [4]:
# Part 2: Knowledge Base

knowledge_base = [
    {"id": "D1", "text": "mia likes apples"},
    {"id": "D2", "text": "sam likes music"},
    {"id": "D3", "text": "tom likes swimming"},
    {"id": "D4", "text": "anna likes painting"},
    {"id": "D5", "text": "lily likes reading"},
]

questions = [
    {"id": "Q1", "question": "what does mia like", "answer": "apples", "doc_id": "D1"},
    {"id": "Q2", "question": "what does sam like", "answer": "music", "doc_id": "D2"},
    {"id": "Q3", "question": "what does tom like", "answer": "swimming", "doc_id": "D3"},
    {"id": "Q4", "question": "what does anna like", "answer": "painting", "doc_id": "D4"},
    {"id": "Q5", "question": "what does lily like", "answer": "reading", "doc_id": "D5"},
]

allowed = set(chars)

for doc in knowledge_base:
    assert set(doc["text"]).issubset(allowed)

for item in questions:
    assert set(item["question"]).issubset(allowed)
    assert set(item["answer"]).issubset(allowed)

print("Knowledge base validated!")
print("Documents:", len(knowledge_base))
print("Questions:", len(questions))

Knowledge base validated!
Documents: 5
Questions: 5


In [5]:
import re

class KnowledgeRetriever:
    def __init__(self, documents):
        self.documents = documents

    def tokenize(self, text):
        return set(re.findall(r"[a-z]+", text.lower()))

    def retrieve(self, query, top_k=1):
        query_words = self.tokenize(query)

        results = []

        for doc in self.documents:
            doc_words = self.tokenize(doc["text"])

            # Word overlap similarity
            score = len(query_words & doc_words)

            results.append({
                "id": doc["id"],
                "text": doc["text"],
                "score": score
            })

        results.sort(
            key=lambda item: item["score"],
            reverse=True
        )

        return results[:top_k]


retriever = KnowledgeRetriever(knowledge_base)

# Verify each question retrieves the correct document first.
for item in questions:
    retrieved = retriever.retrieve(item["question"], top_k=5)

    top_id = retrieved[0]["id"]

    print(
        item["id"],
        "| Expected:", item["doc_id"],
        "| Retrieved:", top_id,
        "| Ranking:", [doc["id"] for doc in retrieved]
    )

    assert top_id == item["doc_id"], (
        f"Retrieval failed for {item['id']}"
    )

print("\nAll five retrieval checks passed!")

Q1 | Expected: D1 | Retrieved: D1 | Ranking: ['D1', 'D2', 'D3', 'D4', 'D5']
Q2 | Expected: D2 | Retrieved: D2 | Ranking: ['D2', 'D1', 'D3', 'D4', 'D5']
Q3 | Expected: D3 | Retrieved: D3 | Ranking: ['D3', 'D1', 'D2', 'D4', 'D5']
Q4 | Expected: D4 | Retrieved: D4 | Ranking: ['D4', 'D1', 'D2', 'D3', 'D5']
Q5 | Expected: D5 | Retrieved: D5 | Ranking: ['D5', 'D1', 'D2', 'D3', 'D4']

All five retrieval checks passed!


In [6]:
# Part 2: No-RAG Baseline
import torch
import pandas as pd

SEED = 42
TEMPERATURE = 1.0
MAX_NEW_TOKENS = 30

@torch.no_grad()
def rag_generate(prompt, seed=SEED):
    # Check that every input character belongs to the model vocabulary.
    unknown = sorted(set(prompt) - set(stoi))
    if unknown:
        raise ValueError(f"Unsupported characters: {unknown}")

    # Fix seed BEFORE each generation for reproducibility.
    torch.manual_seed(seed)

    model.eval()
    context = prompt[-model.context_length:]
    tokens = torch.tensor(
        [[stoi[ch] for ch in context]],
        dtype=torch.long,
        device=DEVICE
    )

    generated = []

    for _ in range(MAX_NEW_TOKENS):
        logits = model(tokens)[:, -1, :]
        probabilities = torch.softmax(
            logits / TEMPERATURE, dim=-1
        )

        next_token = torch.multinomial(
            probabilities, num_samples=1
        )

        char = itos[next_token.item()]
        generated.append(char)

        tokens = torch.cat(
            [tokens, next_token], dim=1
        )[:, -model.context_length:]

    return "".join(generated), context


baseline_results = []

for item in questions:
    response, actual_context = rag_generate(
        item["question"],
        seed=SEED
    )

    baseline_results.append({
        "question_id": item["id"],
        "question": item["question"],
        "expected_answer": item["answer"],
        "actual_context": actual_context,
        "response": response
    })

baseline_df = pd.DataFrame(baseline_results)

display(baseline_df)

,question_id,question,expected_answer,actual_context,response
0,Q1,what does mia like,apples,what does mia like,n.\ntransformers use attention
1,Q2,what does sam like,music,what does sam like,n.\ntransformers use attention
2,Q3,what does tom like,swimming,what does tom like,n.\ntransformers use attention
3,Q4,what does anna like,painting,what does anna like,n.\ntransformers use attention
4,Q5,what does lily like,reading,what does lily like,slanguage models can run on lo


In [7]:
# Part 2: Top-K RAG Experiments

TOP_K_VALUES = [1, 2, 3, 5]

rag_results = []

for item in questions:

    for k in TOP_K_VALUES:

        # 1. Retrieve the top-k documents
        retrieved = retriever.retrieve(
            item["question"],
            top_k=k
        )

        retrieved_ids = [doc["id"] for doc in retrieved]

        # 2. Construct a prompt using retrieved knowledge
        knowledge_text = "\n".join(
            doc["text"] for doc in retrieved
        )

        prompt = (
            knowledge_text
            + "\n"
            + item["question"]
            + " "
        )

        # 3. Generate using the SAME frozen model
        response, actual_context = rag_generate(
            prompt,
            seed=SEED
        )

        # 4. Record results
        rag_results.append({
            "question_id": item["id"],
            "question": item["question"],
            "expected_answer": item["answer"],
            "top_k": k,
            "retrieved_ids": ",".join(retrieved_ids),
            "actual_context": actual_context,
            "context_length": len(actual_context),
            "response": response
        })

rag_df = pd.DataFrame(rag_results)

assert len(rag_df) == 20
assert rag_df["context_length"].le(32).all()

display(rag_df)

print("RAG experiments completed:", len(rag_df))

,question_id,question,expected_answer,top_k,retrieved_ids,actual_context,context_length,response
0,Q1,what does mia like,apples,1,D1,likes apples\nwhat does mia like,32,intelligence.\nstudents create
1,Q1,what does mia like,apples,2,"D1,D2",likes music\nwhat does mia like,32,intelligence.\nstudents create
2,Q1,what does mia like,apples,3,"D1,D2,D3",kes swimming\nwhat does mia like,32,intelligence.\nstudents create
3,Q1,what does mia like,apples,5,"D1,D2,D3,D4,D5",ikes reading\nwhat does mia like,32,intelligence.\nstudents create
4,Q2,what does sam like,music,1,D2,likes music\nwhat does sam like,32,s useful for machine learning
5,Q2,what does sam like,music,2,"D2,D1",likes apples\nwhat does sam like,32,s useful for machine learning
6,Q2,what does sam like,music,3,"D2,D1,D3",kes swimming\nwhat does sam like,32,s useful for machine learning
7,Q2,what does sam like,music,5,"D2,D1,D3,D4,D5",ikes reading\nwhat does sam like,32,s useful for machine learning
8,Q3,what does tom like,swimming,1,D3,kes swimming\nwhat does tom like,32,s can run on local computers.\n
9,Q3,what does tom like,swimming,2,"D3,D1",likes apples\nwhat does tom like,32,s can run on local computers.\n


RAG experiments completed: 20


In [8]:
# Check answer correctness and context visibility

def contains_answer(text, expected):
    return expected.lower() in text.lower()

rag_df["answer_found"] = rag_df.apply(
    lambda row: contains_answer(
        row["response"],
        row["expected_answer"]
    ),
    axis=1
)

rag_df["answer_visible_in_context"] = rag_df.apply(
    lambda row: contains_answer(
        row["actual_context"],
        row["expected_answer"]
    ),
    axis=1
)

print("Correct answers generated:", rag_df["answer_found"].sum(), "/ 20")
print(
    "Expected answer visible in context:",
    rag_df["answer_visible_in_context"].sum(),
    "/ 20"
)

display(
    rag_df[
        [
            "question_id",
            "top_k",
            "retrieved_ids",
            "answer_visible_in_context",
            "answer_found",
            "response"
        ]
    ]
)

Correct answers generated: 0 / 20
Expected answer visible in context: 5 / 20


,question_id,top_k,retrieved_ids,answer_visible_in_context,answer_found,response
0,Q1,1,D1,True,False,intelligence.\nstudents create
1,Q1,2,"D1,D2",False,False,intelligence.\nstudents create
2,Q1,3,"D1,D2,D3",False,False,intelligence.\nstudents create
3,Q1,5,"D1,D2,D3,D4,D5",False,False,intelligence.\nstudents create
4,Q2,1,D2,True,False,s useful for machine learning
5,Q2,2,"D2,D1",False,False,s useful for machine learning
6,Q2,3,"D2,D1,D3",False,False,s useful for machine learning
7,Q2,5,"D2,D1,D3,D4,D5",False,False,s useful for machine learning
8,Q3,1,D3,True,False,s can run on local computers.\n
9,Q3,2,"D3,D1",False,False,s can run on local computers.\n


In [9]:
# Part 2: Quality Scoring

def quality_score(response, expected_answer):
    """
    Reproducible rule-based scoring on a 1-10 scale.
    These are proxy quality scores, not human ratings.
    """
    response = response.lower().strip()
    expected = expected_answer.lower().strip()

    # Correct answer appears as a complete word
    import re
    if re.search(r"\b" + re.escape(expected) + r"\b", response):
        if response == expected:
            return 10
        return 8

    # Partial answer fragment (at least 4 characters)
    if len(expected) >= 4 and expected[:4] in response:
        return 4

    # Text is generated but does not answer the question
    if response and len(response.split()) >= 3:
        return 2

    # Incoherent, incomplete, or nearly empty response
    return 1


baseline_df["quality_score"] = baseline_df.apply(
    lambda row: quality_score(
        row["response"], row["expected_answer"]
    ),
    axis=1
)

rag_df["quality_score"] = rag_df.apply(
    lambda row: quality_score(
        row["response"], row["expected_answer"]
    ),
    axis=1
)

print("No-RAG average:", baseline_df["quality_score"].mean())

print("\nAverage quality by Top-K:")
print(rag_df.groupby("top_k")["quality_score"].mean())

No-RAG average: 2.0

Average quality by Top-K:
top_k
1    2.0
2    2.0
3    2.0
5    2.0
Name: quality_score, dtype: float64


In [10]:
from pathlib import Path

baseline_export = baseline_df.copy()
baseline_export["method"] = "No-RAG"
baseline_export["top_k"] = 0
baseline_export["retrieved_ids"] = ""
baseline_export["context_length"] = baseline_export["actual_context"].str.len()
baseline_export["answer_visible_in_context"] = False

rag_export = rag_df.copy()
rag_export["method"] = rag_export["top_k"].apply(
    lambda k: f"Top-{k} RAG"
)

columns = [
    "method",
    "question_id",
    "question",
    "expected_answer",
    "top_k",
    "retrieved_ids",
    "actual_context",
    "context_length",
    "response",
    "quality_score",
    "answer_visible_in_context"
]

all_results = pd.concat(
    [baseline_export, rag_export],
    ignore_index=True
)[columns]

output_path = Path("a7_part2/part2_rag_results.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

all_results.to_csv(output_path, index=False)

assert len(all_results) == 25
assert all_results["quality_score"].between(1, 10).all()

print("Saved:", output_path)
print("Total rows:", len(all_results))

Saved: a7_part2\part2_rag_results.csv
Total rows: 25


In [11]:
# Part 2: Final RAG Analysis

summary = all_results.groupby("method").agg(
    experiments=("question_id", "count"),
    average_quality=("quality_score", "mean"),
    min_quality=("quality_score", "min"),
    max_quality=("quality_score", "max")
)

print("QUALITY COMPARISON")
display(summary)

# Retrieval ranking accuracy
top1_correct = sum(
    retriever.retrieve(q["question"], top_k=1)[0]["id"] == q["doc_id"]
    for q in questions
)

print(f"\nTop-1 retrieval accuracy: {top1_correct}/5")

# How often does the correct answer survive context truncation?
visibility = rag_df.groupby("top_k")[
    "answer_visible_in_context"
].agg(["sum", "count"])

print("\nANSWER VISIBILITY BY TOP-K")
display(visibility)

# How often did the model generate the correct answer?
print("\nCORRECT ANSWERS BY TOP-K")
display(rag_df.groupby("top_k")["answer_found"].agg(["sum", "count"]))

QUALITY COMPARISON


,experiments,average_quality,min_quality,max_quality
method,,,,
No-RAG,5,2.0,2,2
Top-1 RAG,5,2.0,2,2
Top-2 RAG,5,2.0,2,2
Top-3 RAG,5,2.0,2,2
Top-5 RAG,5,2.0,2,2



Top-1 retrieval accuracy: 5/5

ANSWER VISIBILITY BY TOP-K


,sum,count
top_k,,
1,5,5
2,0,5
3,0,5
5,0,5



CORRECT ANSWERS BY TOP-K


,sum,count
top_k,,
1,0,5
2,0,5
3,0,5
5,0,5


## Part 2: RAG Results and Discussion

### 1. Does RAG improve generation quality compared with the no-RAG baseline?

In this experiment, RAG did not improve generation quality. The no-RAG baseline and all four RAG configurations (Top-1, Top-2, Top-3, and Top-5) received an average rule-based quality score of 2.0 out of 10.

None of the 20 RAG-generated responses contained the expected answer. These results suggest that adding retrieved knowledge alone was insufficient to improve the performance of this small character-level language model.

The quality scores were computed using a reproducible rule-based heuristic rather than human judgments.

### 2. How does the number of retrieved documents affect performance?

Increasing the number of retrieved documents did not improve generation quality.

With Top-1 retrieval, the expected answer remained visible in the model's input for all five questions. However, with Top-2, Top-3, and Top-5 retrieval, the expected answer was no longer visible in any of the five questions.

This happened because the model has a context window of only 32 characters. As additional documents were appended to the prompt, earlier documents were removed through context truncation.

Therefore, retrieving more documents introduced a trade-off between providing more information and preserving relevant information within the model's limited context window.

### 3. Was the retrieval component successful?

Yes. The word-overlap retriever ranked the expected document first for all five questions, achieving 5/5 Top-1 retrieval accuracy.

However, retrieval accuracy did not translate into answer accuracy. This distinction is important because successful retrieval does not guarantee that a language model can interpret and use the retrieved information.

### 4. What were the main limitations of the RAG system?

The main limitations were the model's small capacity, restricted character vocabulary, and short context window.

The pretrained model contained 232,155 parameters, used a vocabulary of 27 characters, and supported a maximum context length of 32 characters. These constraints limited the amount of retrieved information that could be provided to the model.

The generated responses frequently contained generic statements about machine learning or incomplete text rather than answers to the questions.

Additionally, the small evaluation set and heuristic scoring system limit how broadly these findings can be generalized.

### 5. What improvements would you recommend?

The first improvement would be to use a model with a larger context window so that relevant retrieved information is not truncated.

Second, the prompt construction strategy could be improved by placing the most relevant document closest to the question or limiting the amount of retrieved text.

Third, a more capable pretrained language model could improve the system's ability to use retrieved evidence.

Finally, future evaluations should include more questions, more diverse documents, and human quality ratings alongside automatic correctness checks.

Overall, this experiment demonstrates that RAG performance depends on retrieval quality, context management, and the language model's ability to use external knowledge—not retrieval accuracy alone.